# Cyber Surveillance System — Google Colab

هذا الدفتر يشغّل منصة Python المركزية في Colab ويفتح واجهة المختبر عبر رابط HTTPS مؤقت. لا تحتاج إلى تثبيت PyTorch على اللابتوب.

**قبل البدء:** من Runtime → Change runtime type اختر GPU إن كان متاحًا، ثم شغّل الخلايا بالترتيب. عند فتح الواجهة ألصق مفتاح API الذي أدخلته في الدفتر في حقل «مفتاح API» ثم انقر خارج الحقل.

> هذه بيئة اختبار مؤقتة وليست خادمًا دائمًا. رابط النفق عام لمن يملكه؛ واجهات API محمية بمفتاح الجلسة. لا تشارك المفتاح ولا ترفع فيديوهات لا تملك تصريحًا باستخدامها.

In [ ]:
import os, sys, platform, subprocess
print("Python:", sys.version)
print("Platform:", platform.platform())
gpu = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout[:4000] if gpu.returncode == 0 else "No NVIDIA GPU detected; CPU mode is possible but slower.")

## 1. تنزيل المستودع من main

In [ ]:
from pathlib import Path
REPO = Path("/content/Cyber-surveillance-system")
URL = "https://github.com/CybersecurityRahbar/Cyber-surveillance-system.git"
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
%cd /content/Cyber-surveillance-system

## 2. تثبيت الفيديو ومحرك الرؤية

تستخدم هذه الجلسة نسخة PyTorch الموجودة في صورة Colab إن كانت متوافقة، ثم تثبت PyAV وUltralytics. أوزان YOLO26n تُنزّل عند أول تحليل فيديو.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[video]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.197"], check=True)

import torch, av, cv2, ultralytics
print("Torch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("PyAV:", av.__version__, "| OpenCV:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. مكان حفظ الفيديوهات وقاعدة البيانات

افتراضيًا بيانات Colab مؤقتة وتضيع عند انتهاء Runtime. غيّر المتغير إلى True لحفظ البيانات في Google Drive.

In [ ]:
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/CyberSurveillanceLab")
else:
    DATA = Path("/content/cyber-surveillance-data")
(DATA / "lab").mkdir(parents=True, exist_ok=True)
os.environ["TRAFFIC_SURVEILLANCE_DATABASE"] = str(DATA / "surveillance.sqlite3")
os.environ["TRAFFIC_SURVEILLANCE_LAB_STORAGE_DIR"] = str(DATA / "lab")
os.environ["TRAFFIC_SURVEILLANCE_MAX_UPLOAD_MB"] = "90"  # Cloudflare Free limit is 100 MB per request
print("Data directory:", DATA)

## 4. إعداد مفتاح حماية API

أدخل مفتاحًا عشوائيًا خاصًا بك، لا يقل عن 32 حرفًا. ستحتاج إلى إدخال القيمة نفسها في حقل «مفتاح API» في واجهة المتصفح. الإدخال مخفي ولا يُطبع في الدفتر.

In [ ]:
from getpass import getpass
API_KEY = getpass("Private API key (32+ characters): ")
if len(API_KEY) < 32:
    raise ValueError("API key must contain at least 32 characters.")
os.environ["TRAFFIC_SURVEILLANCE_API_KEY"] = API_KEY
os.environ["TRAFFIC_SURVEILLANCE_ENVIRONMENT"] = "development"
os.environ["TRAFFIC_SURVEILLANCE_HOST"] = "127.0.0.1"
print("API protection configured.")

## 5. تهيئة الخدمة

لن نشغّل الخادم أو النفق في خلايا منفصلة. الخلية الأخيرة ستشغّل Uvicorn والنفق معًا، وستظل في حالة Running وتراقب صحتهما. لا تشغّل الخادم يدويًا في خلية أخرى.

In [ ]:
import time, requests, re
from IPython.display import Markdown, display

print("Repository:", REPO)
print("Data directory:", DATA)
print("Python:", sys.version.split()[0])

## 6. تثبيت Cloudflare Tunnel

Quick Tunnel لا يحتاج حسابًا أو نطاقًا. هذه الخلية تثبّت أداة النفق فقط؛ تشغيلها يتم في الخلية الأخيرة حتى لا تنتهي خلية الإعداد بعد طباعة الرابط.

In [ ]:
DEB = "/tmp/cloudflared.deb"
subprocess.run(
    ["bash", "-lc",
     f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O {DEB} && dpkg -i {DEB}"],
    check=True,
)
version = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True, check=True)
print(version.stdout.strip())

## 7. اختيار النفق

Cloudflare هو الخيار الافتراضي. إذا أردت ngrok بدلاً منه، غيّر TUNNEL_PROVIDER إلى ngrok في الخلية التالية؛ سيطلب رمز Auth Token بشكل مخفي.

TUNNEL_PROVIDER = "cloudflare"  # غيّرها إلى ngrok لاستخدام حساب ngrok
NGROK_AUTH_TOKEN = None

if TUNNEL_PROVIDER == "ngrok":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
    NGROK_AUTH_TOKEN = getpass("ngrok Auth Token (hidden): ")
    if len(NGROK_AUTH_TOKEN.strip()) < 20:
        raise ValueError("The ngrok Auth Token looks too short.")
elif TUNNEL_PROVIDER != "cloudflare":
    raise ValueError("TUNNEL_PROVIDER must be cloudflare or ngrok.")

print("Selected tunnel provider:", TUNNEL_PROVIDER)

In [ ]:
# لن يبدأ النفق في هذه الخلية.
# التشغيل الفعلي للخادم والنفق والإشراف يتم في الخلية الأخيرة أسفل الدفتر.
print("Tunnel configuration ready. Run the final supervisor cell.")

## 8. تشغيل الخدمة والإشراف

شغّل الخلية الأخيرة بعد إكمال الإعداد. ستشغّل الخادم المحلي والنفق، وتعرض رابط الواجهة، وتظل نشطة مع فحوص صحة دورية كل 20 ثانية. استخدم زر Interrupt لإيقاف الخلية والخدمات.

هذا يمنع انتهاء خلية التشغيل لمجرد أن الخادم بدأ في الخلفية، لكنه لا يلغي حدود Colab القصوى أو احتمالية قطع Runtime.

In [ ]:
import datetime
from pathlib import Path
from IPython.display import Markdown, display

SERVER_LOG = Path("/tmp/traffic-surveillance-server.log")
server_log_handle = SERVER_LOG.open("w")
server_proc = None
cloudflared_proc = None
ngrok_tunnel = None
PUBLIC_URL = None

def wait_for_server(proc, timeout_seconds=90):
    for _ in range(timeout_seconds):
        if proc.poll() is not None:
            raise RuntimeError(
                "Uvicorn exited unexpectedly:\n" +
                SERVER_LOG.read_text(errors="replace")[-7000:]
            )
        try:
            response = requests.get("http://127.0.0.1:8000/healthz", timeout=3)
            if response.ok:
                return response.json()
        except requests.RequestException:
            pass
        time.sleep(1)
    raise TimeoutError("Uvicorn did not become healthy within the startup window.")

def start_cloudflare():
    global cloudflared_proc
    log_path = Path("/tmp/cloudflared.log")
    log_handle = log_path.open("w")
    cloudflared_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )
    for _ in range(90):
        if cloudflared_proc.poll() is not None:
            raise RuntimeError(
                "cloudflared exited:\n" + log_path.read_text(errors="replace")[-7000:]
            )
        match = re.search(
            r"https://[a-z0-9-]+\\.trycloudflare\\.com",
            log_path.read_text(errors="replace"),
        )
        if match:
            return match.group(0)
        time.sleep(1)
    raise TimeoutError("Cloudflare tunnel URL was not produced.")

def start_ngrok():
    global ngrok_tunnel
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
    ngrok_tunnel = ngrok.connect(8000, "http")
    return ngrok_tunnel.public_url

def stop_services():
    global cloudflared_proc, server_proc
    if cloudflared_proc is not None and cloudflared_proc.poll() is None:
        cloudflared_proc.terminate()
        try:
            cloudflared_proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            cloudflared_proc.kill()
    if server_proc is not None and server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            server_proc.kill()
    if TUNNEL_PROVIDER == "ngrok":
        try:
            from pyngrok import ngrok
            ngrok.kill()
        except Exception:
            pass

# Close old listeners first when rerunning this final cell.
if "cloudflared_proc" in globals() and cloudflared_proc is not None and cloudflared_proc.poll() is None:
    cloudflared_proc.terminate()
if "server_proc" in globals() and server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()

server_proc = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "traffic_surveillance.api.app:app",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--workers", "1",
        "--log-level", "info",
    ],
    cwd=str(REPO),
    env=os.environ.copy(),
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
)

try:
    print("FastAPI ready:", wait_for_server(server_proc))
    PUBLIC_URL = start_cloudflare() if TUNNEL_PROVIDER == "cloudflare" else start_ngrok()
    print("Public tunnel is ready.")
    display(Markdown("## [Open the Lab interface](" + PUBLIC_URL + "/lab)"))
    print("Paste the API key entered earlier into the API key field on the page.")

    status_display = display(
        Markdown("Service active — supervisor cell is running."),
        display_id=True,
    )

    # This is a real supervisor for the server and tunnel, not a busy-loop.
    while True:
        if server_proc.poll() is not None:
            raise RuntimeError(
                "Uvicorn stopped:\n" + SERVER_LOG.read_text(errors="replace")[-7000:]
            )
        if TUNNEL_PROVIDER == "cloudflare" and (
            cloudflared_proc is None or cloudflared_proc.poll() is not None
        ):
            raise RuntimeError(
                "Cloudflare Tunnel stopped:\n" +
                Path("/tmp/cloudflared.log").read_text(errors="replace")[-7000:]
            )
        response = requests.get("http://127.0.0.1:8000/healthz", timeout=5)
        response.raise_for_status()
        stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        status_display.update(
            Markdown(
                "Service active · " + TUNNEL_PROVIDER +
                " · Last health check: " + stamp +
                " · Supervisor cell is still running."
            )
        )
        time.sleep(20)

except KeyboardInterrupt:
    print("Interrupt received. Stopping the server and tunnel...")
finally:
    stop_services()
    print("Session services stopped. Rerun the final cell to start again.")